# CHSH-style EstimatorV2 sweep

Evaluate two Bell-pair correlations across a measurement-basis sweep with Qiskit and MettleQ estimators.

The SDK reference and MettleQ calls below use the same circuit and result contract. Timing includes the complete call shown.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

from mettleq.integrations.qiskit import (
    MettleQBackend,
    MettleQEstimatorV2,
    MettleQSamplerV2,
)
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    phase_aligned_statevector_error,
    qiskit_selection,
    total_variation_distance,
)

In [ ]:
angles = np.linspace(0.0, np.pi, 9)
observables = [SparsePauliOp("ZZ"), SparsePauliOp("ZX")]
circuits = []
for angle in angles:
    circuit = QuantumCircuit(2)
    circuit.h(0)
    circuit.cx(0, 1)
    circuit.ry(float(angle), 0)
    circuits.append(circuit)
pubs = [(circuit, observables) for circuit in circuits]

def run_reference():
    return np.asarray([StatevectorEstimator().run([pub]).result()[0].data.evs for pub in pubs])

reference, reference_ms, _ = benchmark(run_reference)
backend = MettleQBackend(method="statevector", device="cpu")
mettleq_pubs = [(transpile(circuit, backend, optimization_level=1), observables) for circuit in circuits]
estimator = MettleQEstimatorV2(backend=backend)

def run_mettleq():
    return np.asarray([estimator.run([pub]).result()[0].data.evs for pub in mettleq_pubs])

candidate, mettleq_ms, _ = benchmark(run_mettleq)
error = max_abs_error(reference, candidate)
method, device = qiskit_selection(estimator)
tutorial_result = emit_result(
    notebook="qiskit/04_estimator_chsh.ipynb",
    framework="qiskit",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="EstimatorV2 correlation curve atol=2e-6",
    passed=error <= 2e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_expectation_error": error, "angles": angles, "reference": reference, "mettleq": candidate},
)